# ForgeSight — Fine-tune Qwen2.5-3B on a free Colab T4

Trains the QLoRA fine-tune end-to-end on a **free Colab T4 (16 GB)** and downloads the GGUF you
then serve locally via Ollama.

**Setup:**
1. `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.
2. `Runtime → Run all`.

> Colab runs on a remote VM and can't read `C:\...` paths, so **cell 2 opens an upload dialog**.
> Pick either **`finetune.zip`** (built from the repo) or the 3 loose files
> (`colab_train.py`, `sft_train.jsonl`, `sft_eval.jsonl`). Run-all will pause there for the upload.

Pipeline: upload files → install Unsloth → train (`colab_train.py`) → export `Q4_K_M.gguf` →
download. Locally: `ollama create qwen-forgesight -f finetune/Modelfile`.

In [1]:
#@title 1. Confirm the T4 GPU is attached
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [2]:
#@title 2. Upload finetune.zip (or the 3 loose training files)
# Colab runs on a remote VM, so it can't read C:\... paths — upload instead. Pick EITHER:
#   (a) finetune.zip, OR
#   (b) the 3 loose files: colab_train.py, sft_train.jsonl, sft_eval.jsonl
import os, zipfile, shutil
from pathlib import Path
from google.colab import files

if os.path.isdir("/content/finetune"):
    shutil.rmtree("/content/finetune")
os.makedirs("finetune/dataset", exist_ok=True)

print("Choose finetune.zip (or the 3 loose files)…")
uploaded = files.upload()

# Stage the upload (zip contents + any loose files) into one temp tree.
stage = Path("_staged")
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir()
for fn in list(uploaded):
    if fn.lower().endswith(".zip"):
        with zipfile.ZipFile(fn) as z:
            z.extractall(stage)
        os.remove(fn)
    elif os.path.exists(fn):
        shutil.move(fn, stage / os.path.basename(fn))

# Match required files by basename, normalizing separators (a PowerShell-made zip can store
# Windows backslashes, so Linux sees "dataset\sft_train.jsonl" as one flat filename).
targets = {
    "colab_train.py":  "finetune/colab_train.py",
    "sft_train.jsonl": "finetune/dataset/sft_train.jsonl",
    "sft_eval.jsonl":  "finetune/dataset/sft_eval.jsonl",
}
found = {p.name.replace("\\", "/").split("/")[-1]: p
         for p in stage.rglob("*") if p.is_file()}
for name, dest in targets.items():
    if name in found:
        shutil.copy(found[name], dest)

missing = [d for d in targets.values() if not os.path.exists(d)]
if missing:
    print("Could not place:", missing)
    print("Files seen in your upload:", [p.name for p in stage.rglob("*") if p.is_file()])
shutil.rmtree(stage, ignore_errors=True)
assert not missing, f"Still missing: {missing}. Check the list above, then re-run this cell."
print("\nReady:")
!wc -l finetune/dataset/sft_train.jsonl finetune/dataset/sft_eval.jsonl

Choose files to upload (a zip of finetune/, or the 3 loose files)...


Saving finetune.zip to finetune.zip

Ready:
   35 finetune/dataset/sft_train.jsonl
    3 finetune/dataset/sft_eval.jsonl
   38 total


In [3]:
#@title 3. Install Unsloth + training stack (~2-3 min)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
#@title 4. Train + export GGUF (T4: max_seq=2048, bsz 2x4, 2 epochs, lr=1e-4)
# Building llama.cpp for the GGUF export adds a few minutes the first time.
!python finetune/colab_train.py --data finetune/dataset/sft_train.jsonl

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[1/5] loading unsloth/Qwen2.5-3B-Instruct (4-bit, max_seq=2048) …
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 434/434 [00:01<00:00, 299.03it/s]
unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[2/5] applying LoRA adapters (r=16, gradient checkpointing=unsloth) …
Unsloth 2026.6.7 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
[3/5] preparing dataset 

In [5]:
#@title 5. Download the GGUF as qwen-forgesight.Q4_K_M.gguf
import glob, os, shutil
from google.colab import files

cands = glob.glob("finetune/export/**/*.gguf", recursive=True)
assert cands, "No GGUF found — check the training cell output for export errors."
src = max(cands, key=os.path.getsize)               # the Q4_K_M weights file
out = "/content/qwen-forgesight.Q4_K_M.gguf"        # canonical name (matches the Modelfile)
shutil.copy(src, out)
print(f"Source export : {src}")
print(f"Downloading   : {out}  ({os.path.getsize(out)/1e9:.2f} GB) …")
files.download(out)

Downloading: finetune/export/qwen-forgesight_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf (1.93 GB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# (Optional) confirm the file that was prepared for download:
!ls -lh /content/qwen-forgesight.Q4_K_M.gguf

## Back on your laptop (RTX 3060 + Ollama)

The file downloads (and is saved to your Drive) as **`qwen-forgesight.Q4_K_M.gguf`**. Put it at the
exact path the `Modelfile` points to:

```bash
# place file at: finetune/export/qwen-forgesight.Q4_K_M.gguf
ollama create qwen-forgesight -f finetune/Modelfile

# measure base vs fine-tune (JSON validity · intent · citation compliance):
python finetune/03_evaluate_vs_base.py --model qwen2.5:3b-instruct
python finetune/03_evaluate_vs_base.py --model qwen-forgesight

# promote ONLY if the fine-tune wins: set OLLAMA_MODEL=qwen-forgesight and
# SYNTHESIS_BACKEND=ollama in your local .env.
```

Record the eval JSON in `docs/finetune.md`. The public Railway URL keeps using Groq (no cloud GPU);
the fine-tuned SLM is the on-prem local path.